![BeliefLens](assets/belieflens-notebook-header.png)

**BeliefLens access required:** [Request a BeliefLens API key](https://demo.belieflens.org/signup). Keep the key in the `BELIEFLENS_API_KEY` environment variable; never paste it into this notebook.

# Apply a frozen BeliefLens measurement profile from LangChain

## Goal

Insert an already validated BeliefLens measurement into an existing LangChain or LangGraph workflow. A single `measurement_profile_id` identifies the frozen benchmark version, ontology, prompt family, model identity, JSON calibrator and acceptance certificate. LangChain invokes the profile; it does not refit it.

## What happens

1. Inspect the benchmark schema so the measured states are explicit.
2. Load the frozen measurement profile by identifier.
3. Submit one evidence record through a LangChain-compatible node.
4. Receive calibrated state probabilities, uncertainty checks, a certificate decision and a trace link.
5. Route the next graph action to continue, review or abstain.

Code inputs are hidden by default; click a cell's disclosure control to inspect them. Outputs remain visible.

**Further information:** [BeliefLens primer](https://belieflens.org/#/primer) · [Software and integration guide](https://belieflens.org/#/software) · [Benchmark Lab](https://belieflens.org/#/benchmarks)


## 1. Inspect the declared benchmark schema

The local manifest makes the scientific task visible. At runtime the server verifies the corresponding frozen profile and applies its bound calibrator. The calibrator is persisted as schema-versioned JSON coefficients inside a content-hashed validation artifact—not as an executable pickle.


In [1]:
import json, os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = Path('examples/notebooks/finance')
manifest = json.loads((ROOT / 'data/offline_reproduction/inputs/SPY_SGOV_benchmark_manifest.json').read_text())
print('Benchmark schema:', manifest.get('schema_version'))
print('Declared states:', manifest.get('states') or manifest.get('ontology', {}).get('states'))


Benchmark schema: belieflens-benchmark-manifest-v1
Declared states: [{'id': 'risk_on', 'value': 'Risk-on', 'definition': 'Evidence is collectively consistent with a constructive broad-equity environment, such as positive equity momentum and breadth, contained stress, or supportive credit conditions.'}, {'id': 'mixed', 'value': 'Mixed', 'definition': 'Evidence is conflicting, transitional, or insufficiently one-sided to support either the Risk-on or Risk-off state.'}, {'id': 'risk_off', 'value': 'Risk-off', 'definition': 'Evidence is collectively consistent with a defensive or stressed broad-equity environment, such as negative momentum or breadth, elevated volatility, or deteriorating credit conditions.'}]


### Use a dedicated BeliefLens–LangChain kernel

The SDK is supplied by the neighbouring `BeliefBench-Server` repository rather than the public Python package index. Installing LangChain into a large shared environment can change transitive packages such as `websockets` and conflict with unrelated applications. The recommended setup is therefore a small isolated kernel. Run these commands in a terminal from this reproducibility repository—not in a notebook cell:

```bash
python3 -m venv .venv-belieflens-langchain
source .venv-belieflens-langchain/bin/activate
python -m pip install --upgrade pip
python -m pip install -e "../BeliefBench-Server/sdk/python[langchain]" ipykernel pandas matplotlib
python -m ipykernel install --user --name belieflens-langchain --display-name "BeliefLens + LangChain"
```

Then select **Kernel → Change Kernel → BeliefLens + LangChain** in Jupyter and restart this notebook. If the two repositories are not siblings, replace the relative SDK path with its absolute path, for example:

```bash
python -m pip install -e "/Users/matthewdixon_1/Downloads/codex-workspace/papers/BeliefBench-Server/sdk/python[langchain]"
```

The `[langchain]` suffix selects optional dependencies; it must remain inside the quoted path. The next cell verifies the selected kernel but deliberately installs nothing.


In [2]:
import importlib.metadata

try:
    from belieflens import BeliefLens
    from belieflens.integrations.langchain import BeliefLensMeasurementRunnable
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'BeliefLens is not installed in this Jupyter kernel. Follow the dedicated-kernel instructions above.'
    ) from exc

print('BeliefLens', importlib.metadata.version('belieflens'))
print('LangChain Core', importlib.metadata.version('langchain-core'))
print('BeliefLens SDK and LangChain integration are available in this kernel.')


BeliefLens 0.1.0
LangChain Core 0.3.86
BeliefLens SDK and LangChain integration are available in this kernel.


In [3]:
%pip install -e "/Users/matthewdixon_1/Downloads/codex-workspace/papers/BeliefBench-Server/sdk/python[langchain]"

Obtaining file:///Users/matthewdixon_1/Downloads/codex-workspace/papers/BeliefBench-Server/sdk/python


  Installing build dependencies ... -

 \

 done


  Checking if build backend supports build_editable ... done


...

  Preparing editable metadata (pyproject.toml) ... - done


  Building editable for belieflens (pyproject.toml) ... - done
  Created wheel for belieflens: filename=belieflens-0.1.0-0.editable-py3-none-any.whl size=4209 sha256=ee28fb9f40ff26b05d3e2929544c0852f928c9bfde94069929a597ba58484d7b
  Stored in directory: /private/var/folders/xj/47dwgr1d5jd7l2rygr10vjxr0000gp/T/pip-ephem-wheel-cache-b_nd_zio/wheels/c9/6d/f0/b2b7c20d2d04d32b7a3164f67adb4fb1c67e75d23ba09e18ea
Successfully built belieflens


  Attempting uninstall: belieflens
    Found existing installation: belieflens 0.1.0
    Uninstalling belieflens-0.1.0:


      Successfully uninstalled belieflens-0.1.0

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [4]:
try:
    import belieflens
    from belieflens.integrations.langchain import BeliefLensMeasurementRunnable
    print('BeliefLens SDK and LangChain integration are available.')
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "BeliefLens is not installed. Until the PyPI release, maintainers should install "
        "the adjacent source checkout shown above."
    ) from exc


BeliefLens SDK and LangChain integration are available.


In [5]:
MEASUREMENT_PROFILE_ID = os.getenv(
    'BELIEFLENS_MEASUREMENT_PROFILE_ID',
    'cert_showcase_us_equity',  # Toy financial profile; override for your validated profile.
)
required = ['BELIEFLENS_API_KEY', 'OPENAI_API_KEY']
live_ready = all(os.getenv(name) for name in required)
client = measurement = profile = None
if live_ready:
    from belieflens import BeliefLens
    from belieflens.integrations.langchain import BeliefLensMeasurementRunnable
    client = BeliefLens(
        base_url=os.getenv('BELIEFLENS_BASE_URL', 'https://demo.belieflens.org'),
        api_key=os.environ['BELIEFLENS_API_KEY'],
    )
    measurement = BeliefLensMeasurementRunnable(
        client=client,
        measurement_profile_id=MEASUREMENT_PROFILE_ID,
        provider_api_key=os.environ['OPENAI_API_KEY'],
    )
    profile = client.measurement_profile(MEASUREMENT_PROFILE_ID)
    print('Live profile loaded:', profile['id'], 'hash:', profile['content_hash'])
else:
    print('Offline demonstration mode. Missing:', ', '.join(name for name in required if not os.getenv(name)))


Offline demonstration mode. Missing: BELIEFLENS_API_KEY, OPENAI_API_KEY


## 3. Invoke it as a chain or LangGraph node

This is calibration **application**, not calibration fitting. One call returns calibrated state probabilities, uncertainty diagnostics, a certificate decision and a trace identifier. Set `RUN_PROVIDER_CALLS=1` only when you intend to make the provider calls required by the frozen prompt family.


In [6]:
record = {
    'evidence_text': 'As of 2018-01-10, SPY gained 1.3% over five days and 4.0% over twenty; volatility was 7.0%, breadth was broad, and credit conditions were supportive.',
    'observation_time': '2018-01-10T21:00:00Z',
}
if live_ready and os.getenv('RUN_PROVIDER_CALLS') == '1':
    result = measurement.invoke(record)
else:
    result = {
        'schema_version': 'belieflens-runtime-measurement-v1',
        'measurement_id': 'archived-example-daily-2018-01-10',
        'state_probabilities': {'Risk-on': 0.6308, 'Mixed': 0.3454, 'Risk-off': 0.0238},
        'decision': 'measurement_valid_state_ambiguous',
        'observability': {
            'trace_id': None, 'trace_url': None,
            'stages': ['provider measurement', 'semantic map', 'calibration', 'runtime checks', 'certificate decision'],
            'note': 'Archived offline example; no live trace was emitted.',
        },
    }
print(json.dumps({
    'state_probabilities': result['state_probabilities'],
    'decision': result['decision'],
    'measurement_id': result['measurement_id'],
    'observability': result['observability'],
}, indent=2))


{
  "state_probabilities": {
    "Risk-on": 0.6308,
    "Mixed": 0.3454,
    "Risk-off": 0.0238
  },
  "decision": "measurement_valid_state_ambiguous",
  "measurement_id": "archived-example-daily-2018-01-10",
  "observability": {
    "trace_id": null,
    "trace_url": null,
    "stages": [
      "provider measurement",
      "semantic map",
      "calibration",
      "runtime checks",
      "certificate decision"
    ],
    "note": "Archived offline example; no live trace was emitted."
  }
}


## 4. Gate the next action

In LangGraph, use the returned decision before a trade, external tool or other consequential node. The gate does not claim that the proposed action is correct; it establishes whether the semantic measurement passed its declared conditions.


In [7]:
def route_measurement(state):
    decision = state['measurement']['decision']
    if decision == 'measurement_accepted_within_scope':
        return 'continue'
    if 'abstain' in decision:
        return 'abstain'
    return 'human_review'

print('Next workflow route:', route_measurement({'measurement': result}))

# graph.add_conditional_edges('measure', route_measurement, {
#     'continue': 'action', 'human_review': 'review', 'abstain': 'stop'
# })


Next workflow route: human_review
